In [67]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

In [68]:
from skbio.stats.composition import ancombc
from sklearn.model_selection import StratifiedKFold

import pandas as pd
import numpy as np


from scipy.stats import kstest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import chi2, norm
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import argparse
from statistics import mean

In [69]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, balanced_accuracy_score
)

def binary_metrics(y_true, y_pred, pos_label="Urban"):
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    precision = precision_score(y_true, y_pred, pos_label=pos_label)
    recall = recall_score(y_true, y_pred, pos_label=pos_label)
    accuracy = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[label for label in sorted(set(y_true)) if label != pos_label] + [pos_label]
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "Accuracy": accuracy,
        "Balanced Accuracy": bal_acc,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    }

In [70]:
data = pd.read_csv('../datasets/latinbiota_merge_metaphlan_data.csv', sep = '\t', skiprows=1, index_col = 0)
metadata = pd.read_excel('../data/metadata_LATINBIOTA_MEXICO.xlsx', sheet_name='Data')

In [71]:
data_norm = pd.read_csv('metaphlan_norm.csv', index_col = 0)

In [72]:
data_norm.head()

,Bacteroides,Bifidobacterium,Phocaeicola,Fusicatenibacter,Roseburia,Hominilimicola,Blautia,Faecalibacterium,GGB4605,Ruminococcus,...,Sphingomonas,GGB109105,GGB142492,GGB9763,GGB101460,GGB100295,GGB79087,GGB18111,GGB13711,Pauljensenia
36703_3#10,32.43933,11.20012,10.17307,6.61526,5.25075,4.15566,3.92776,3.55883,3.39242,2.86484,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36703_3#12,0.00000,2.08148,0.00000,1.63042,2.77936,0.02314,2.39343,13.65037,0.00000,11.70037,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36703_3#13,0.01499,1.26326,0.00000,0.05598,3.73623,0.00000,0.46372,20.11008,0.00000,6.14148,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36703_3#14,0.01129,3.93269,0.02558,0.32470,0.16471,0.08696,1.64632,0.50796,0.08947,2.75049,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36703_3#15,0.00000,14.90514,0.00000,0.60660,0.09425,1.44976,3.14182,0.48589,0.10213,4.45759,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [73]:
data_genus = pd.read_csv('metaphlan_clr_genus_wo_zeros_150_pseudo.csv', index_col = 0)

In [74]:
data_genus

,Bacteroides,Bifidobacterium,Phocaeicola,Fusicatenibacter,Roseburia,Hominilimicola,Blautia,Faecalibacterium,GGB4605,Ruminococcus,...,Phoenicibacter,GGB9420,GGB9347,Treponema,GGB9762,GGB9345,GGB9787,Hominifimenecus,GGB9189,otros
36703_3#10,32.439303,11.200093,10.173043,6.615233,5.250723,4.155633,3.927733,3.558803,3.392393,2.864813,...,0.000008,0.000008,0.000008,0.000008,0.000008,0.000008,0.000008,0.000008,0.000008,1.460753
36703_3#12,0.000019,2.081464,0.000019,1.630404,2.779344,0.023124,2.393414,13.650354,0.000019,11.700354,...,0.000019,0.000019,0.000019,0.000019,0.000019,0.000019,0.000019,0.000019,0.000019,17.297894
36703_3#13,0.014969,1.263239,0.000014,0.055959,3.736209,0.000014,0.463699,20.110059,0.000014,6.141459,...,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,15.003629
36703_3#14,0.011266,3.932666,0.025556,0.324676,0.164686,0.086936,1.646296,0.507936,0.089446,2.750466,...,0.000011,0.000011,0.000011,0.000011,0.000011,0.000011,0.000011,0.000011,0.000011,18.267816
36703_3#15,0.000009,14.905114,0.000009,0.606574,0.094224,1.449734,3.141794,0.485864,0.102104,4.457564,...,0.000009,0.000009,0.000009,0.000009,0.000009,0.000009,0.000009,0.000009,0.000009,4.854584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37082_3#5,9.401228,1.025178,0.604268,0.718018,1.987978,0.000023,0.768958,3.417898,0.019318,5.398058,...,0.009968,0.013248,0.000978,0.000023,0.004708,0.011788,0.024238,0.000023,0.000023,7.992868
37082_3#6,0.009149,6.393249,0.000014,0.034389,2.831709,0.210849,8.968199,21.198609,0.075809,4.456209,...,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,0.000014,5.931789
37082_3#7,0.000026,0.252731,0.000026,0.035551,0.000941,0.008371,1.027491,0.390471,0.022871,0.482441,...,0.000026,0.033831,0.019531,0.881041,0.007301,0.448041,0.142131,0.000201,0.003351,44.482461
37082_3#8,0.000028,0.129933,0.000028,0.128473,0.072583,0.000028,2.291293,3.343453,0.022733,0.820953,...,0.000028,0.078913,0.061063,0.000028,0.092763,0.071073,0.237403,0.000028,0.006963,32.185843


In [75]:
labels  = []
for id_ in data_genus.index:
    labels.append(metadata[metadata['Lane'] == id_]['Lifestyle'].iloc[0])

In [76]:
labels = np.asarray(labels)

assert len(data_genus) == len(labels)
assert (data_genus >= 0).all().all()

In [77]:
mask = data_genus.sum(axis=1) > 0
data_genus = data_genus.loc[mask]
labels = labels[mask.values]

In [78]:
metadata_order = pd.DataFrame(
    {"Lifestyle": labels},
    index=data_genus.index
)

In [79]:
metadata_order

,Lifestyle
36703_3#10,Urban
36703_3#12,Rural
36703_3#13,Rural
36703_3#14,Rural
36703_3#15,Rural
...,...
37082_3#5,Rural
37082_3#6,Rural
37082_3#7,Rural
37082_3#8,Rural


In [80]:
result = ancombc(data_genus, metadata_order, formula='Lifestyle')

In [81]:
result

Log2(FC)        SE          W  \
FeatureID       Covariate                                            
Bacteroides     Intercept           -1.219623  0.336234  -3.627308   
                Lifestyle[T.Urban]  10.379627  0.458945  22.616288   
Bifidobacterium Intercept            4.317883  0.453811   9.514713   
                Lifestyle[T.Urban]   3.461963  0.627128   5.520345   
Phocaeicola     Intercept           -2.050272  0.294470  -6.962582   
...                                       ...       ...        ...   
Hominifimenecus Lifestyle[T.Urban]   3.722483  0.419475   8.874149   
GGB9189         Intercept           -2.583664  0.180579 -14.307690   
                Lifestyle[T.Urban]   3.655405  0.395314   9.246840   
otros           Intercept           10.486237  0.118329  88.619480   
                Lifestyle[T.Urban]   1.057909  0.176319   5.999988   

                                           pvalue         qvalue  Signif  
FeatureID       Covariate                                                 
Bacteroides     Intercept            2.863912e-04   2.405686e-02    True  
                Lifestyle[T.Urban]  2.996571e-113  5.903244e-111    True  
Bifidobacterium Intercept            1.822154e-21   2.715010e-19    True  
                Lifestyle[T.Urban]   3.383357e-08   3.992361e-06    True  
Phocaeicola     Intercept            3.340930e-12   4.176162e-10    True  
...                                           ...            ...     ...  
Hominifimenecus Lifestyle[T.Urban]   7.047011e-19   1.099334e-16    True  
GGB9189         Intercept            1.959124e-46   3.271738e-44    True  
                Lifestyle[T.Urban]   2.312280e-20   3.676526e-18    True  
otros           Intercept            0.000000e+00   0.000000e+00    True  
                Lifestyle[T.Urban]   1.973319e-09   2.446916e-07    True  

[400 rows x 6 columns]

In [82]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [83]:
def ks_test(df, healthy, non_healthy, method_ks = 'asymp', p_val = 0.001):
    healthy_df = df[[x for x in df.columns if x in healthy]].T
    nonhealthy_df = df[[x for x in df.columns if x in non_healthy]].T

    # print(healthy_df)
    # print(nonhealthy_df)
    
    healthy_features = []
    nonhealthy_features = []
    for feature in list(df.index):
        if kstest(list(healthy_df[feature]), list(nonhealthy_df[feature]), alternative = 'less', method = method_ks).pvalue <= p_val:
            healthy_features.append(feature)
        if kstest(list(nonhealthy_df[feature]), list(healthy_df[feature]), alternative = 'less', method = method_ks).pvalue <= p_val:
            nonhealthy_features.append(feature)
    # print(f'# Healthy features selected by KS: {len(healthy_features)}')
    # print(f'# Unheatlhy features selected by KS: {len(nonhealthy_features)}')
    return healthy_features, nonhealthy_features

In [84]:
urban = list(metadata[metadata['Lifestyle'] == 'Urban']['Lane'])
rural = list(metadata[metadata['Lifestyle'] == 'Rural']['Lane'])
# non_healthy = list(metadata[metadata_train[args.diagnosis] != args.control][args.sample])    

In [85]:
def custom_transform(x):
    if x <= 1:
        return np.log2(2 * x + 0.00001)
    else:
        return np.sqrt(x)

def transform_data(df, features):
    scaler = StandardScaler()
    aux = pd.DataFrame()
    for item in list(set(features)):
        if item in df.index:
            aux[item] = list(df.T[item])
        else:
            aux[item] = [0 for x in range(len(df.T))]
    selected = aux.applymap(custom_transform)

    scaler.fit(selected)
    # with open(f'model_data/{model_name}/scaler.pkl', 'wb') as file:
    #     pickle.dump(scaler, file)
    selected2 = scaler.transform(selected)

    scaling_data = pd.DataFrame(zip(selected.columns, scaler.mean_, scaler.scale_), columns = ['specie', 'mean', 'std'])
    
    
    # pd.DataFrame(zip(selected.columns, scaler.mean_, scaler.scale_), columns = ['specie', 'mean', 'std']).to_csv(f'model_data/{model_name}/scaling_parameters.csv', index = False)


    # for c in selected.columns:
    #     scaler.fit(np.array(selected[c]).reshape(-1, 1))
    #     selected[c] = scaler.transform(np.array(selected[c]).reshape(-1, 1))
        # params = params.append({'mean':scaler.mean_[0], 'std':scaler.scale_[0]}, ignore_index=True)
        # print(scaler.mean_)
        # print(scaler.scale_)
    # print(selected)
    selected2 = pd.DataFrame(selected2, columns = selected.columns)
    selected2.index = df.T.index

    return selected2, scaling_data, scaler
    
def transform_data3(df, features):
    scaler = StandardScaler()
    aux = pd.DataFrame()
    for item in list(set(features)):
        if item in df.index:
            aux[item] = list(df.T[item])
        else:
            aux[item] = [0 for x in range(len(df.T))]
    selected = aux

    scaler.fit(selected)
    # with open(f'model_data/{model_name}/scaler.pkl', 'wb') as file:
    #     pickle.dump(scaler, file)
    selected2 = scaler.transform(selected)

    scaling_data = pd.DataFrame(zip(selected.columns, scaler.mean_, scaler.scale_), columns = ['specie', 'mean', 'std'])
    
    
    # pd.DataFrame(zip(selected.columns, scaler.mean_, scaler.scale_), columns = ['specie', 'mean', 'std']).to_csv(f'model_data/{model_name}/scaling_parameters.csv', index = False)


    # for c in selected.columns:
    #     scaler.fit(np.array(selected[c]).reshape(-1, 1))
    #     selected[c] = scaler.transform(np.array(selected[c]).reshape(-1, 1))
        # params = params.append({'mean':scaler.mean_[0], 'std':scaler.scale_[0]}, ignore_index=True)
        # print(scaler.mean_)
        # print(scaler.scale_)
    # print(selected)
    selected2 = pd.DataFrame(selected2, columns = selected.columns)
    selected2.index = df.T.index

    return selected2, scaling_data, scaler
    
def transform_data2(df, features):
    # scaler = StandardScaler()
    aux = pd.DataFrame()
    for item in list(set(features)):
        if item in df.index:
            aux[item] = list(df.T[item])
        else:
            aux[item] = [0 for x in range(len(df.T))]
    selected = aux

    return selected


def calculate_pca_stats(df, variance_for_pc = 0.9, alpha = 0.05):
    pca = PCA()

    # print(df[df.isna().any(axis=1)])
    # print(df)

    pca.fit(df)

    # with open(f'model_data/{model_name}/pca_model.pkl', 'wb') as file:
    #     pickle.dump(pca, file)

    eigenvalues = pca.explained_variance_
    eigenvectors = pca.components_
    singular = pca.singular_values_
    
    pca_data = pd.DataFrame(zip(eigenvectors, eigenvalues, singular), columns = ('Eigenvectors', 'Explained_variance', 'Singular_values')).sort_values('Explained_variance', ascending = False)
    pca_data['%variance'] = pca_data['Explained_variance'] / sum(pca_data['Explained_variance'])
    pca_data = pca_data.sort_values('%variance', ascending = False)
    pca_data['%variance_cumulative'] = pca_data['%variance'].cumsum()
    
    principal_components = list(pca_data[pca_data['%variance_cumulative'] < variance_for_pc]['Eigenvectors'])
    # print(f'# Principal Components selected: {len(principal_components)}')
    
    principal_values = list(pca_data[pca_data['%variance_cumulative'] < variance_for_pc]['Explained_variance'])
    D = np.array(principal_components).T @ np.linalg.inv(np.diag(principal_values)) @ np.array(principal_components)
    deg_free = len(principal_components) 
    # alpha = 0.05
    t2_threshold = chi2.ppf(1-alpha, deg_free)
#     print(1-alpha, deg_free)
#     print(t2_threshold)
    
    principal_components_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Eigenvectors'])
    principal_values_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Explained_variance'])
    principal_singvalues_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Singular_values'])
    
    C = np.array(principal_components_residual).T @ np.array(principal_components_residual)
    Theta1 = sum(principal_values_residual)
    Theta2 = sum([x**2 for x in principal_values_residual])
    Theta3 = sum([x**3 for x in principal_values_residual])
    
    c_alpha = norm.ppf(1-alpha)
    
    h0 = 1-((2*Theta1*Theta3)/(3*(Theta2**2)))

    '''INTENTO'''
    Q_alpha = Theta1*(((((c_alpha*np.sqrt(2*Theta2*(h0**2)))/Theta1)+1+((Theta2*h0*(h0-1))/(Theta1**2))))**(1/h0))
    # print(Q_alpha)
    Q_alpha = Theta1*(((((np.sqrt(c_alpha*(2*Theta2*(h0**2))))/Theta1)+1-((Theta2*h0*(h0-1))/(Theta1**2))))**(1/h0))
    # print(Q_alpha)
    Q_alpha = (Theta2/Theta1) * chi2.ppf(alpha, len(principal_components_residual)) * ((Theta1**2)/Theta2)
    # print(Q_alpha)
    
    # Q_alpha = Theta1*(((((c_alpha*np.sqrt(2*Theta2*(h0**2)))/Theta1)+1+((Theta2*h0*(h0-1))/(Theta1**2))))**(1/h0))
    
    #fi = D/t2_threshold + (np.eye(len(principal_components[0])) - (np.array(principal_components).T @ np.array(principal_components)))/Q_alpha
    fi = D/t2_threshold  + C/Q_alpha
    g = ((len(principal_components) / t2_threshold**2) + (Theta2 / Q_alpha**2)) / ((len(principal_components)/t2_threshold) + (Theta1 / Q_alpha))
    h = ((len(principal_components)/t2_threshold) + (Theta1 / Q_alpha))**2 / ((len(principal_components) / t2_threshold**2) + (Theta2 / Q_alpha**2))

    chi_value = chi2.ppf(1-alpha, h)
    threshold_combined = g*chi_value
    
    return pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined

def hotelling_t2(df, pca, pca_data, variance_for_pc = 0.9, alpha = 0.05):
    principal_components = list(pca_data[pca_data['%variance_cumulative'] < variance_for_pc]['Eigenvectors'])
    # print(f'# Principal Components selected: {len(principal_components)}')
    principal_values = list(pca_data[pca_data['%variance_cumulative'] < variance_for_pc]['Explained_variance'])
    D = np.array(principal_components).T @ np.linalg.inv(np.diag(principal_values)) @ np.array(principal_components)
    deg_free = len(principal_components) 
    # alpha = 0.05
    t2_threshold = chi2.ppf(1-alpha, deg_free)
    T2 = []
    pred = []
    
    try:
        for item in pca.transform(df):
            index = item.T @ D @ item
            T2.append(index)
            if index > t2_threshold:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')
    except:
        for item in np.array(df):
            index = item.T @ D @ item
            T2.append(index)
            if index > t2_threshold:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')
            
    hoteling = pd.DataFrame(zip(df.index, T2, pred), columns = ['Sample', 'T2', 'Prediction T2'])
    
    return D, principal_components, hoteling, t2_threshold

def Q_statistic(df, pca, pca_data, variance_for_pc = 0.9, alpha = 0.05):
    principal_components_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Eigenvectors'])
    principal_values_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Explained_variance'])
    principal_singvalues_residual = list(pca_data[pca_data['%variance_cumulative'] >= variance_for_pc]['Singular_values'])
    
    C = np.array(principal_components_residual).T @ np.array(principal_components_residual)
    Theta1 = sum(principal_values_residual)
    Theta2 = sum([x**2 for x in principal_values_residual])
    Theta3 = sum([x**3 for x in principal_values_residual])
    
    c_alpha = norm.ppf(1-alpha)
    
    h0 = 1-((2*Theta1*Theta3)/(3*Theta2**2))

    #! NO BORRAR ORIGINAL
    Q_alpha = Theta1*(((((c_alpha*np.sqrt(2*Theta2*(h0**2)))/Theta1)+1+((Theta2*h0*(h0-1))/(Theta1**2))))**(1/h0))
    # print(Q_alpha)
    Q_alpha = Theta1*(((((np.sqrt(c_alpha*(2*Theta2*(h0**2))))/Theta1)+1-((Theta2*h0*(h0-1))/(Theta1**2))))**(1/h0))
    # print(Q_alpha)
    Q_alpha = (Theta2/Theta1) * chi2.ppf(1-alpha, len(principal_components_residual)) * ((Theta1**2)/Theta2)
    # print(Q_alpha)
    
    Q = []
    pred = []
    try:
        for item in pca.transform(df):
            index = item.T @ C @ item
            Q.append(index)
            if index > Q_alpha:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')
    except:
        for item in np.array(df):
            index = item.T @ C @ item
            Q.append(index)
            if index > Q_alpha:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')
    
    Q_statistic = pd.DataFrame(zip(df.index, Q, pred), columns = ['Sample', 'Q', 'Prediction Q'])
    
    return C, Theta1, Theta2, Q_statistic, Q_alpha
    

def combined_index(df, D, t2_threshold, principal_components, Q_alpha, Theta1, Theta2, pca, alpha = 0.05):
    fi = D/t2_threshold + (np.eye(len(principal_components[0])) - (np.array(principal_components).T @ np.array(principal_components)))/Q_alpha
    g = ((len(principal_components) / t2_threshold**2) + (Theta2 / Q_alpha**2)) / ((len(principal_components)/t2_threshold) + (Theta1 / Q_alpha))
    h = ((len(principal_components)/t2_threshold) + (Theta1 / Q_alpha))**2 / ((len(principal_components) / t2_threshold**2) + (Theta2 / Q_alpha**2))

    chi_value = chi2.ppf(1-alpha, h)
    threshold_combined = g*chi_value
    combined = []
    pred = []

    try:
        for item in pca.transform(df):
            index = item.T @ fi @ item
            combined.append(index)
            if index > threshold_combined:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')
    except:
        for item in np.array(df):
            index = item.T @ fi @ item
            combined.append(index)
            if index > threshold_combined:
                pred.append('Unhealthy')
            else:
                pred.append('Healthy')

    combined = pd.DataFrame(zip(df.index, combined, pred), columns = ['Sample', 'Combined', 'Prediction Combined']) 
    return combined

def hiPCA(df, healthy, non_healthy, features = [], ks = False, method = 'auto', p_val = 0.001, only_nonhealthy_features = False):
    if ks:
        healthy_features, non_healthy_features = ks_test(df, healthy, non_healthy, method_ks = method, p_val = p_val)
        
    if only_nonhealthy_features:
        healthy_features = []
        if ks:
            features = healthy_features + non_healthy_features
        selected, scaling_data, scaler = transform_data(df[[x for x in healthy if x in df.columns]], features)
        # selected = transform_data(df[[x for x in healthy if x in df.columns]], features)
        
    else:
        if ks:
            features = healthy_features + non_healthy_features
        selected, scaling_data, scaler = transform_data(df[[x for x in healthy if x in df.columns]], features)
        # selected = transform_data(df[[x for x in healthy if x in df.columns]], features)
    
    # print(selected)
    pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined = calculate_pca_stats(selected)
    # np.save(f'model_data/{model_name}/D_matrix.npy', D)
    # np.save(f'model_data/{model_name}/C_matrix.npy', C)
    # np.save(f'model_data/{model_name}/fi_matrix.npy', fi)

    thresholds = {'t2':t2_threshold, 'c':Q_alpha, 'combined':threshold_combined}
    

    # with open(f'model_data/{model_name}/thresholds.json', 'w') as json_file:
    #     json.dump(thresholds, json_file)

    # print(t2_threshold, Q_alpha, threshold_combined)

        
    return features, pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined, selected, scaling_data, scaler
    
def hiPCA2(df, healthy, non_healthy, features = [], ks = False, method = 'auto', p_val = 0.001, only_nonhealthy_features = False):
    if ks:
        healthy_features, non_healthy_features = ks_test(df, healthy, non_healthy, method_ks = method, p_val = p_val)
        
    if only_nonhealthy_features:
        healthy_features = []
        if ks:
            features = healthy_features + non_healthy_features
        selected, scaling_data, scaler = transform_data3(df[[x for x in healthy if x in df.columns]], features)
        # selected = transform_data2(df[[x for x in healthy if x in df.columns]], features)
        
    else:
        if ks:
            features = healthy_features + non_healthy_features
        selected, scaling_data, scaler = transform_data3(df[[x for x in healthy if x in df.columns]], features)
        # selected = transform_data2(df[[x for x in healthy if x in df.columns]], features)
    
    # print(selected)
    pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined = calculate_pca_stats(selected)
    # np.save(f'model_data/{model_name}/D_matrix.npy', D)
    # np.save(f'model_data/{model_name}/C_matrix.npy', C)
    # np.save(f'model_data/{model_name}/fi_matrix.npy', fi)

    thresholds = {'t2':t2_threshold, 'c':Q_alpha, 'combined':threshold_combined}
    

    # with open(f'model_data/{model_name}/thresholds.json', 'w') as json_file:
    #     json.dump(thresholds, json_file)

    # print(t2_threshold, Q_alpha, threshold_combined)

        
    return features, pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined, selected, scaling_data, scaler

def transform_data_evaluate(df, scaling_data, scaler):
    # scaling_data = pd.read_csv(f'{path}/scaling_parameters.csv')
    features = list(scaling_data['specie'])
    # print(df.index)
    # with open(f'{path}/scaler.pkl', 'rb') as file:
    #     scaler = pickle.load(file)
    # print(features)
    # scaler = StandardScaler()
    aux = pd.DataFrame()
    for item in list(set(features)):
        # print(item)
        if item in df.index:
            aux[item] = list(df.T[item])
        else:
            aux[item] = [0 for x in range(len(df.T))]
    # print(aux)
    selected = aux.applymap(custom_transform)

    # scaler.fit(np.array(selected))
    # print(selected)
    selected = selected[features]
    # print(selected)
    selected2 = scaler.transform(selected)
    selected2 = pd.DataFrame(selected2, columns = selected.columns)
    selected2.index = df.T.index
    

    return selected2
def transform_data_evaluate2(df, scaling_data, scaler):
    # scaling_data = pd.read_csv(f'{path}/scaling_parameters.csv')
    features = list(scaling_data['specie'])
    # print(df.index)
    # with open(f'{path}/scaler.pkl', 'rb') as file:
    #     scaler = pickle.load(file)
    # print(features)
    # scaler = StandardScaler()
    aux = pd.DataFrame()
    for item in list(set(features)):
        # print(item)
        if item in df.index:
            aux[item] = list(df.T[item])
        else:
            aux[item] = [0 for x in range(len(df.T))]
    # print(aux)
    selected = aux

    # scaler.fit(np.array(selected))
    # print(selected)
    selected = selected[features]
    # print(selected)
    selected2 = scaler.transform(selected)
    selected2 = pd.DataFrame(selected2, columns = selected.columns)
    selected2.index = df.T.index
    

    return selected2
    
# def transform_data_evaluate2(df, features):
    # scaling_data = pd.read_csv(f'{path}/scaling_parameters.csv')
    # features = list(scaling_data['specie'])
    # print(df.index)
    # with open(f'{path}/scaler.pkl', 'rb') as file:
    #     scaler = pickle.load(file)
    # print(features)
    # scaler = StandardScaler()
    # aux = pd.DataFrame()
    # for item in list(set(features)):
    #     # print(item)
    #     if item in df.index:
    #         aux[item] = list(df.T[item])
    #     else:
    #         aux[item] = [0 for x in range(len(df.T))]
    # # print(aux)
    # selected = aux.applymap(custom_transform)

    # # scaler.fit(np.array(selected))
    # # print(selected)
    # selected = selected[features]
    # # print(selected)
    # selected2 = scaler.transform(selected)
    # selected2 = pd.DataFrame(selected2, columns = selected.columns)
    # selected2.index = df.T.index
    

    # return selected2

def calculate_index(data_transformed, C, D, fi, pca, t2_threshold, Q_alpha, threshold_combined):
    # D = np.load(f'{path}/D_matrix.npy')
    # C = np.load(f'{path}/C_matrix.npy')
    # fi = np.load(f'{path}/fi_matrix.npy')
    # pca = joblib.load(f'{path}/pca_model.pkl')


    # with open(f'{path}/thresholds.json', 'r') as file:
    #     thresholds = json.load(file)
    #     t2_threshold = thresholds['t2']
    #     Q_alpha = thresholds['c']
    #     threshold_combined = thresholds['combined']

    T2, Q, combined = [], [], []
    pred_t2, pred_Q, pred_combined = [], [], []

    try:
        for item in pca.transform(data_transformed):
            index = item.T @ D @ item
            index2 = item.T @ C @ item
            index3 = item.T @ fi @ item
            T2.append(index)
            Q.append(index2)
            combined.append(index3)
            if index > t2_threshold:
                pred_t2.append('Urban')
            else:
                pred_t2.append('Rural')

            if index2 > Q_alpha:
                pred_Q.append('Urban')
            else:
                pred_Q.append('Rural')

            if index3 > threshold_combined:
                pred_combined.append('Urban')
            else:
                pred_combined.append('Rural') 
    except:
        for item in np.array(data_transformed):
            index = item.T @ D @ item
            index2 = item.T @ C @ item
            index3 = item.T @ fi @ item
            T2.append(index)
            Q.append(index2)
            combined.append(index3)
            if index > t2_threshold:
                pred_t2.append('Urban')
            else:
                pred_t2.append('Rural')

            if index2 > Q_alpha:
                pred_Q.append('Urban')
            else:
                pred_Q.append('Rural')

            if index3 > threshold_combined:
                pred_combined.append('Urban')
            else:
                pred_combined.append('Rural') 

    return pd.DataFrame(zip(data_transformed.index, T2, pred_t2, Q, pred_Q, combined, pred_combined), columns = ['SampleID', 'T2', 'Prediction T2', 'Q', 'Prediction Q', 'Combined Index', 'Combined Prediction'])

In [86]:
def clr_transform(X):
    """
    Apply Centered Log-Ratio (CLR) transformation to compositional data.
    
    Parameters:
    X : array-like, shape (n_samples, n_features)
        Compositional data (must be positive)
    
    Returns:
    X_clr : array, CLR-transformed data
    """
    # Add small constant to avoid log(0) if there are zeros
    X = X + 1e-6
    
    # Compute geometric mean for each sample (row)
    geom_mean = np.exp(np.mean(np.log(X), axis=1, keepdims=True))
    
    # CLR: log(x_i / geometric_mean)
    X_clr = np.log(X / geom_mean)
    
    return X_clr

In [87]:
import statistics

In [89]:
results = []
original = []
proposed = []
for fold, (train_idx, test_idx) in enumerate(skf.split(data_genus, metadata_order['Lifestyle']), start=1):
    X_train = data_genus.iloc[train_idx]
    X_train2 = data_norm.iloc[train_idx]
    
    X_test = data_genus.iloc[test_idx]
    X_test2 = data_norm.iloc[test_idx]

    y_train = metadata_order.iloc[train_idx]
    y_test = metadata_order.iloc[test_idx]

    # print(list(y_test['Lifestyle']))

    

    # healthy_features, non_healthy_features = ks_test(X_train.T, rural, urban, method_ks = 'asymp', p_val = 0.001)
    X = clr_transform(X_train.values)
    new_X = pd.DataFrame(X, columns = X_train.columns, index = X_train.index)

    # print(new_X)
    # print(new_X.T[new_X.T.isna().any(axis=1)])

    features, pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined, selected, scaling, scaler = hiPCA2(
        new_X.T, rural, urban, features = new_X.columns, ks = True, method = 'exact', only_nonhealthy_features = False
    )

   
    X_t = clr_transform(X_test.values)
    X_t = pd.DataFrame(X_t, columns = X_test.columns, index = X_test.index)

    X_t = X_t[features]

    data_evaluate = transform_data_evaluate2(X_t.T, scaling, scaler)
    # print(data_evaluate)

    results_n = calculate_index(data_evaluate, C, D, fi, pca, t2_threshold, Q_alpha, threshold_combined)
    results_n['TRUE'] = list(y_test['Lifestyle'])
    # print(results)

    features, pca, pca_data, D, t2_threshold, C, Q_alpha, fi, threshold_combined, selected, scaling, scaler = hiPCA(
        X_train2.T, rural, urban, ks = True, method = 'exact', only_nonhealthy_features = True
    )

    # print(len(X_test2))
    data_evaluate = transform_data_evaluate(X_test2.T, scaling, scaler)
            # print(data)
            # data.to_csv('transformed.csv')
    results_o = calculate_index(data_evaluate, C, D, fi, pca, t2_threshold, Q_alpha, threshold_combined)
    # print(len(results))
    results_o['TRUE'] = list(y_test['Lifestyle'])
    # print(results)

    results_n['original'] = list(results_o['Combined Prediction'])
    # print(results_n)

    metrics_original = binary_metrics(
        results_n['TRUE'],
        results_n['original'],
        pos_label="Urban"
    )
    
    # Propuesto
    metrics_proposed = binary_metrics(
        results_n['TRUE'],
        results_n['Combined Prediction'],
        pos_label="Urban"
    )
    
    original.append(metrics_original["F1"])
    proposed.append(metrics_proposed["F1"])
    
    print("=== Original ===")
    for k, v in metrics_original.items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
    
    print("\n=== Proposed ===")
    for k, v in metrics_proposed.items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
    
    # f1_urban = f1_score(results_n['TRUE'], results_n['Prediction T2'], pos_label="Rural")
    # # proposed.append(f1_urban)
    # print('Prop:', f1_urban)

    
    # print(X_t)
    
print(statistics.mean(original))
print(statistics.mean(proposed))
    

    # print(D, C)
    # print(healthy_features, non_healthy_features)

    # print(y_train)
    # print(X_train)

    # result = ancombc(X_train, y_train, formula='Lifestyle')
    # print(len(result.query('Covariate != "Intercept" & Signif == True')))
    # grouping must be pandas Series aligned with samples
    # grouping = pd.Series(y_train, index=X_train.index, name="group")

    # # ANCOM-BC expects samples x features
    # res = ancombc(
    #     table=X_train,
    #     grouping=grouping,
    #     formula="group",
    #     p_adjust="fdr_bh",
    #     alpha=0.05,
    #     max_iter=100,
    #     tol=1e-5,
    # )

    # # res is an AncomBCResults object
    # res_df = res.results.copy()

    # res_df["taxon"] = res_df.index
    # res_df["fold"] = fold
    # res_df["significant"] = res_df["reject"]

    # results.append(res_df)


=== Original ===
F1: 0.718
Precision: 0.636
Recall: 0.824
Specificity: 0.652
Accuracy: 0.725
Balanced Accuracy: 0.738
TP: 14
FP: 8
TN: 15
FN: 3

=== Proposed ===
F1: 1.000
Precision: 1.000
Recall: 1.000
Specificity: 1.000
Accuracy: 1.000
Balanced Accuracy: 1.000
TP: 17
FP: 0
TN: 23
FN: 0
=== Original ===
F1: 0.650
Precision: 0.565
Recall: 0.765
Specificity: 0.565
Accuracy: 0.650
Balanced Accuracy: 0.665
TP: 13
FP: 10
TN: 13
FN: 4

=== Proposed ===
F1: 0.938
Precision: 1.000
Recall: 0.882
Specificity: 1.000
Accuracy: 0.950
Balanced Accuracy: 0.941
TP: 15
FP: 0
TN: 23
FN: 2
=== Original ===
F1: 0.683
Precision: 0.560
Recall: 0.875
Specificity: 0.542
Accuracy: 0.675
Balanced Accuracy: 0.708
TP: 14
FP: 11
TN: 13
FN: 2

=== Proposed ===
F1: 0.968
Precision: 1.000
Recall: 0.938
Specificity: 1.000
Accuracy: 0.975
Balanced Accuracy: 0.969
TP: 15
FP: 0
TN: 24
FN: 1
=== Original ===
F1: 0.737
Precision: 0.636
Recall: 0.875
Specificity: 0.667
Accuracy: 0.750
Balanced Accuracy: 0.771
TP: 14
FP: 8
